In [1]:
import pandas as pd
import numpy as np
import Bio
from Bio.Restriction import AllEnzymes
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 加载数据

In [2]:
# Load data files
df_methylation = pd.read_csv('./utils/output/methylation_check.csv')
df_seamless = pd.read_csv('./utils/output/restriction_enzyme_seamless_insert.csv')
df_silent = pd.read_csv('./utils/output/restriction_enzyme_slient_mutation.csv')

print(f"Methylation check: {df_methylation.shape}")
print(f"Seamless insert: {df_seamless.shape}")
print(f"Silent mutation: {df_silent.shape}")

Methylation check: (6064, 11)
Seamless insert: (14180, 14)
Silent mutation: (7727, 15)


## 辅助函数

In [3]:
def check_methylation_compatible(enzyme_name, df_methylation):
    """Check if enzyme is not sensitive to 6mA/5mC methylation (DH5α compatible)"""
    dh5a_compatible = set(
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'enzyme'].dropna().tolist() +
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'prototype'].dropna().tolist()
    )
    return enzyme_name in dh5a_compatible

def get_enzyme_info(enzyme_name):
    """Get detailed information about an enzyme"""
    try:
        enzyme = getattr(Bio.Restriction, enzyme_name)
        site = str(enzyme.site)
        site_length = len(enzyme.site)
        fst5 = enzyme.fst5
        fst3 = enzyme.fst3
        ovhg = enzyme.ovhg
        
        # 判断切割类型
        is_type_iis = fst5 > site_length or fst3 > 0  # fst3通常是负数，正数表示切在外面
        cuts_outside = fst5 > site_length
        
        return {
            'site': site,
            'site_length': site_length,
            'fst5': fst5,
            'fst3': fst3,
            'ovhg': ovhg,
            'cuts_outside_5': cuts_outside,
            'is_type_iis': is_type_iis
        }
    except Exception as e:
        return None

## Site I 候选酶分析（Seamless Insert）

In [4]:
# Get Site I candidates
site_i_enzymes = df_seamless['name'].unique()
print(f"Total enzymes in seamless insert file: {len(site_i_enzymes)}")

# Filter by methylation
site_i_filtered = [e for e in site_i_enzymes if check_methylation_compatible(e, df_methylation)]
print(f"After methylation filter: {len(site_i_filtered)}")

# Get detailed info
site_i_info = []
for enzyme in site_i_filtered:
    info = get_enzyme_info(enzyme)
    if info:
        info['enzyme'] = enzyme
        site_i_info.append(info)

df_site_i = pd.DataFrame(site_i_info)
print(f"\nSite I candidates with enzyme info: {len(df_site_i)}")
print(f"\nCut types distribution:")
print(df_site_i['is_type_iis'].value_counts())
print(df_site_i['cuts_outside_5'].value_counts())

Total enzymes in seamless insert file: 70
After methylation filter: 70

Site I candidates with enzyme info: 70

Cut types distribution:
is_type_iis
False    70
Name: count, dtype: int64
cuts_outside_5
False    70
Name: count, dtype: int64


In [5]:
# Display Site I candidates
print("\n=" * 80)
print("SITE I CANDIDATES (Seamless Insert)")
print("=" * 80)
display(df_site_i.sort_values('enzyme'))


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
SITE I CANDIDATES (Seamless Insert)


,site,site_length,fst5,fst3,ovhg,cuts_outside_5,is_type_iis,enzyme
0,GACGTC,6,5,-5,4,False,False,AatII
1,CCTCGAGG,8,2,-2,-4,False,False,AbsI
2,CCGC,4,1,-1,-2,False,False,AciI
3,AACGTT,6,2,-2,-2,False,False,AclI
4,CTTAAG,6,1,-1,-4,False,False,AflII
5,ACCGGT,6,1,-1,-4,False,False,AgeI
6,GGGCCC,6,5,-5,4,False,False,ApaI
7,GTGCAC,6,1,-1,-4,False,False,ApaLI
8,GGCGCGCC,8,2,-2,-4,False,False,AscI
9,GGTACC,6,1,-1,-4,False,False,Asp718I


## Site II 候选酶分析（Silent Mutation）

In [6]:
# Get Site II candidates
site_ii_enzymes = df_silent['name'].unique()
print(f"Total enzymes in silent mutation file: {len(site_ii_enzymes)}")

# Filter by methylation
site_ii_filtered = [e for e in site_ii_enzymes if check_methylation_compatible(e, df_methylation)]
print(f"After methylation filter: {len(site_ii_filtered)}")

# Get detailed info
site_ii_info = []
for enzyme in site_ii_filtered:
    info = get_enzyme_info(enzyme)
    if info:
        info['enzyme'] = enzyme
        site_ii_info.append(info)

df_site_ii = pd.DataFrame(site_ii_info)
print(f"\nSite II candidates with enzyme info: {len(df_site_ii)}")
print(f"\nCut types distribution:")
print(df_site_ii['is_type_iis'].value_counts())
print(df_site_ii['cuts_outside_5'].value_counts())

Total enzymes in silent mutation file: 57
After methylation filter: 57

Site II candidates with enzyme info: 57

Cut types distribution:
is_type_iis
False    57
Name: count, dtype: int64
cuts_outside_5
False    57
Name: count, dtype: int64


In [7]:
# Display Site II candidates
print("\n=" * 80)
print("SITE II CANDIDATES (Silent Mutation)")
print("=" * 80)
display(df_site_ii.sort_values('enzyme'))


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
SITE II CANDIDATES (Silent Mutation)


,site,site_length,fst5,fst3,ovhg,cuts_outside_5,is_type_iis,enzyme
0,CCTCGAGG,8,2,-2,-4,False,False,AbsI
1,CCGC,4,1,-1,-2,False,False,AciI
2,AACGTT,6,2,-2,-2,False,False,AclI
3,CTTAAG,6,1,-1,-4,False,False,AflII
4,ACCGGT,6,1,-1,-4,False,False,AgeI
5,GTGCAC,6,1,-1,-4,False,False,ApaLI
6,GGCGCGCC,8,2,-2,-4,False,False,AscI
7,GGTACC,6,1,-1,-4,False,False,Asp718I
8,CCTAGG,6,1,-1,-4,False,False,AvrII
9,GGATCC,6,1,-1,-4,False,False,BamHI


## Site III 候选酶分析

**需要确定：Site III应该满足什么条件？**

可能的理解：
1. Type IIS酶（fst5 > site_length）- 切割在识别位点外
2. 与Site II不同的酶，但产生相同overhang
3. 其他特殊条件？

让我们检查所有可能的候选：

In [8]:
# Site III也来自silent mutation，但需要特殊条件
print("检查不同切割类型的酶数量：")
print(f"\nTotal silent mutation enzymes: {len(df_site_ii)}")
print(f"  - cuts_outside_5=True (fst5 > site_length): {df_site_ii['cuts_outside_5'].sum()}")
print(f"  - cuts_outside_5=False (fst5 <= site_length): {(~df_site_ii['cuts_outside_5']).sum()}")
print(f"  - is_type_iis=True: {df_site_ii['is_type_iis'].sum()}")
print(f"  - is_type_iis=False: {(~df_site_ii['is_type_iis']).sum()}")

检查不同切割类型的酶数量：

Total silent mutation enzymes: 57
  - cuts_outside_5=True (fst5 > site_length): 0
  - cuts_outside_5=False (fst5 <= site_length): 57
  - is_type_iis=True: 0
  - is_type_iis=False: 57


In [9]:
# 检查fst5和fst3的分布
print("\nfst5 distribution:")
print(df_site_ii['fst5'].value_counts().sort_index())

print("\nfst3 distribution:")
print(df_site_ii['fst3'].value_counts().sort_index())


fst5 distribution:
fst5
1    38
2    13
3     2
4     2
5     2
Name: count, dtype: int64

fst3 distribution:
fst3
-5     2
-4     2
-3     2
-2    13
-1    38
Name: count, dtype: int64


In [10]:
# 显示几个典型例子
print("\n=" * 80)
print("典型例子")
print("=" * 80)

print("\n1. 常规酶 (fst5 <= site_length):")
display(df_site_ii[~df_site_ii['cuts_outside_5']].head(10))

print("\n2. Type IIS酶 (fst5 > site_length):")
type_iis_examples = df_site_ii[df_site_ii['cuts_outside_5']]
if len(type_iis_examples) > 0:
    display(type_iis_examples.head(10))
else:
    print("没有找到Type IIS酶！")


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
典型例子

1. 常规酶 (fst5 <= site_length):


,site,site_length,fst5,fst3,ovhg,cuts_outside_5,is_type_iis,enzyme
0,CCTCGAGG,8,2,-2,-4,False,False,AbsI
1,CCGC,4,1,-1,-2,False,False,AciI
2,AACGTT,6,2,-2,-2,False,False,AclI
3,CTTAAG,6,1,-1,-4,False,False,AflII
4,ACCGGT,6,1,-1,-4,False,False,AgeI
5,GTGCAC,6,1,-1,-4,False,False,ApaLI
6,GGCGCGCC,8,2,-2,-4,False,False,AscI
7,GGTACC,6,1,-1,-4,False,False,Asp718I
8,CCTAGG,6,1,-1,-4,False,False,AvrII
9,GGATCC,6,1,-1,-4,False,False,BamHI



2. Type IIS酶 (fst5 > site_length):
没有找到Type IIS酶！


## Overhang分析

Site II和Site III需要产生相同的overhang用于连接重复单元

In [11]:
print("Overhang分布（Silent Mutation酶）:")
print(df_site_ii['ovhg'].value_counts().sort_index())

print("\n每个overhang类型的酶数量：")
for ovhg in sorted(df_site_ii['ovhg'].unique()):
    enzymes_with_ovhg = df_site_ii[df_site_ii['ovhg'] == ovhg]
    print(f"  ovhg={ovhg:>3}: {len(enzymes_with_ovhg)} enzymes")
    if len(enzymes_with_ovhg) > 1:
        print(f"      可用于Site II/III配对: {list(enzymes_with_ovhg['enzyme'].values[:5])}...")

Overhang分布（Silent Mutation酶）:
ovhg
-4    36
-3     1
-2    14
 2     6
Name: count, dtype: int64

每个overhang类型的酶数量：
  ovhg= -4: 36 enzymes
      可用于Site II/III配对: ['AbsI', 'AflII', 'AgeI', 'ApaLI', 'AscI']...
  ovhg= -3: 1 enzymes
  ovhg= -2: 14 enzymes
      可用于Site II/III配对: ['AciI', 'AclI', 'BfaI', 'BspDI', 'BstBI']...
  ovhg=  2: 6 enzymes
      可用于Site II/III配对: ['BstKTI', 'HhaI', 'PacI', 'PvuI', 'RgaI']...


## 切割位点详细分析

理解fst5和fst3的含义：
- fst5: 5'链上的切割位置（从5'端开始计数）
- fst3: 3'链上的切割位置（通常是负数，从3'端开始计数）

示例：
```
EcoRI: GAATTC (site_length=6)
       G^AATTC  (fst5=1, 在第1个碱基后切割)
       CTTAA^G  (fst3=-1, 从末尾倒数第1个碱基前切割)
```

In [12]:
# 选择几个代表性的酶展示切割模式
examples = [
    'EcoRI',   # 经典常规酶
    'BamHI',   # 经典常规酶
    'BsaI',    # 经典Type IIS
    'PstI',    # 常规酶
    'SmaI',    # 平末端
]

print("\n代表性酶的切割模式：")
print("=" * 80)
for enzyme_name in examples:
    info = get_enzyme_info(enzyme_name)
    if info:
        print(f"\n{enzyme_name}:")
        print(f"  Recognition site: {info['site']} (length={info['site_length']})")
        print(f"  Cut positions: fst5={info['fst5']}, fst3={info['fst3']}")
        print(f"  Overhang: {info['ovhg']}")
        print(f"  Cuts outside 5' end: {info['cuts_outside_5']}")
        print(f"  Type IIS: {info['is_type_iis']}")


代表性酶的切割模式：

EcoRI:
  Recognition site: GAATTC (length=6)
  Cut positions: fst5=1, fst3=-1
  Overhang: -4
  Cuts outside 5' end: False
  Type IIS: False

BamHI:
  Recognition site: GGATCC (length=6)
  Cut positions: fst5=1, fst3=-1
  Overhang: -4
  Cuts outside 5' end: False
  Type IIS: False

BsaI:
  Recognition site: GGTCTC (length=6)
  Cut positions: fst5=7, fst3=5
  Overhang: -4
  Cuts outside 5' end: True
  Type IIS: True

PstI:
  Recognition site: CTGCAG (length=6)
  Cut positions: fst5=5, fst3=-5
  Overhang: 4
  Cuts outside 5' end: False
  Type IIS: False

SmaI:
  Recognition site: CCCGGG (length=6)
  Cut positions: fst5=3, fst3=-3
  Overhang: 0
  Cuts outside 5' end: False
  Type IIS: False


## 总结与建议

**请根据上面的信息回答以下问题：**

1. Site I应该选择哪些类型的酶？（目前：seamless insert中的酶，methylation compatible）

2. Site II应该选择哪些类型的酶？（目前：silent mutation中的酶，methylation compatible）

3. **Site III应该选择哪些类型的酶？请明确指定判断条件：**
   - 选项A：Type IIS酶（fst5 > site_length）
   - 选项B：与Site II相同pool，但不同酶，相同overhang
   - 选项C：其他条件？

4. "识别位点和切割位点不重叠"具体指什么？

In [13]:
# 保存结果供参考
df_site_i.to_csv('output/site_i_candidates_detail.csv', index=False)
df_site_ii.to_csv('output/site_ii_candidates_detail.csv', index=False)

print("\n候选酶详细信息已保存：")
print("  - output/site_i_candidates_detail.csv")
print("  - output/site_ii_candidates_detail.csv")


候选酶详细信息已保存：
  - output/site_i_candidates_detail.csv
  - output/site_ii_candidates_detail.csv
